# Train RayGNN from scratch on Kaggle

Select two **GPUs** in Kaggle Settings, enable Internet for the setup cell, and attach the `linrock/nn-335a9b2d8a80-t80-oct2022-bestmove` binpack dataset. This is a two-GPU screening run using the [second RayGNN design](RayGNN_Design_Document_2nd.md). It initializes a new model and uses the teacher scores embedded in each binpack record. No NNUE checkpoint is loaded.

The setup cell updates an existing checkout to the latest `raygnn` branch before importing it. If you previously ran an older checkout in this Kaggle session, restart the kernel before rerunning the notebook so Python loads the updated modules and native library. One binpack is enough to train; validation is used only when a separate second binpack is attached. The screening target is about 20 million sampled positions, matching the design document's 20 one-million-position epochs. Validation and checkpoints run every 102,400 positions, so partial runs still show progress. The direct FEN encoder reduces CPU overhead, and the batch is split across two GPUs. This is a teacher-score fit screen; measuring playing strength requires engine games and a baseline. Input files stay read-only in `/kaggle/input`; this run writes checkpoints to `/kaggle/working/raygnn_run_2gpu` to preserve earlier runs.


In [ ]:
import random
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU, then restart the session.'
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT == 2, f'Expected two Kaggle GPUs, found {GPU_COUNT}.'
DEVICE = torch.device('cuda:0')
SEED = 17
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('PyTorch:', torch.__version__)
print('Training GPUs:', [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])


In [ ]:
# Clone or update the branch containing RayGNN and scored-FEN loader changes.
import subprocess
from pathlib import Path

REPO = Path('/kaggle/working/nnue-pytorch')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'raygnn',
                    'https://github.com/lualum/nnue-pytorch.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'raygnn'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'raygnn'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)
SOURCE_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', SOURCE_COMMIT)
import sys
loaded = any(name == package or name.startswith(package + '.')
             for name in sys.modules for package in ('raygnn', 'data_loader'))
if loaded and globals().get('_RAYGNN_LOADED_COMMIT') != SOURCE_COMMIT:
    raise RuntimeError('Restart the Kaggle kernel, then run the notebook from the top. '
                       'Python still has RayGNN or the native loader from an earlier checkout loaded.')
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'python-chess==0.31.4'], check=True)
subprocess.run(['cmake', '-S', str(REPO / 'data_loader/cpp'), '-B', str(REPO / 'build'),
                '-DCMAKE_BUILD_TYPE=Release', f'-DLIB_COPY_DIR={REPO}'], check=True)
subprocess.run(['cmake', '--build', str(REPO / 'build'), '-j4'], check=True)

import os, sys
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from raygnn import RayGNN, RayGNNConfig, RayGNNEvaluator, fens_to_batch
from data_loader import FenBatchProvider
from data_loader._native import FenBatch
assert RayGNN().head[0].in_features == 976, 'The cloned branch needs the second-design RayGNN changes.'
assert hasattr(FenBatch, 'get_fens_and_scores'), 'The cloned branch needs the scored-FEN loader changes.'
_RAYGNN_LOADED_COMMIT = SOURCE_COMMIT


In [ ]:
# Read binpacks in place. A second file provides an independent validation stream.
binpacks = sorted(Path('/kaggle/input').rglob('*.binpack'), key=lambda p: p.stat().st_size, reverse=True)
if not binpacks:
    raise FileNotFoundError('Attach a Kaggle dataset containing at least one .binpack file.')
TRAIN_BINPACK = binpacks[0]
VAL_BINPACK = binpacks[1] if len(binpacks) > 1 else None
for path in binpacks:
    with path.open('rb') as handle:
        header = handle.read(4)
    if header != b'BINP':
        raise ValueError(f'Invalid binpack header: {path}')
    print(path, f'{path.stat().st_size / 2**30:.2f} GiB')
print('Train:', TRAIN_BINPACK)
print('Validation:', VAL_BINPACK or 'disabled: no separate binpack')


In [ ]:
# The binpack score is side-to-move positive in Stockfish internal units.
# The loader's win-rate model uses 208 units per pawn, so convert to White-positive pawn units.
import chess
from torch.nn import functional as F

BATCH_SIZE = 128 * GPU_COUNT
SCREENING_POSITIONS = 20_000_000
STEPS_PER_INTERVAL = 800 // GPU_COUNT
POSITIONS_PER_INTERVAL = BATCH_SIZE * STEPS_PER_INTERVAL
INTERVALS = (SCREENING_POSITIONS + POSITIONS_PER_INTERVAL - 1) // POSITIONS_PER_INTERVAL
TOTAL_POSITIONS = INTERVALS * POSITIONS_PER_INTERVAL
SCHEDULE_INTERVALS = (50_000_000 + POSITIONS_PER_INTERVAL - 1) // POSITIONS_PER_INTERVAL
VALIDATION_STEPS = 50
LEARNING_RATE = 3e-4
RUN_DIR = Path('/kaggle/working/raygnn_run_2gpu')
RUN_DIR.mkdir(parents=True, exist_ok=True)

def make_stream(path, cyclic):
    return FenBatchProvider(str(path), cyclic=cyclic, num_workers=2,
                            batch_size=BATCH_SIZE, include_scores=True)

def encode_scored_batch(fens, scores):
    rows = [(fen, score) for fen, score in zip(fens, scores)
            if abs(score) < 30000]  # omit mate-like labels
    if not rows:
        return None
    fens, scores = zip(*rows)
    batch = fens_to_batch(list(fens))
    stm = batch.side_to_move[:, 0]
    target = torch.tensor(scores, dtype=torch.float32) * stm / 208.0
    return batch, target[:, None].to(DEVICE)

@torch.inference_mode()
def validation_huber(network):
    if VAL_BINPACK is None:
        return None
    stream = make_stream(VAL_BINPACK, cyclic=False)
    network.eval()
    total, count = torch.zeros((), device=DEVICE), 0
    try:
        for _ in range(VALIDATION_STEPS):
            try:
                sample = encode_scored_batch(*next(stream))
            except StopIteration:
                break
            if sample is None:
                continue
            batch, target = sample
            prediction = network(batch.piece, batch.side_to_move, batch.castling, batch.en_passant)
            total += F.huber_loss(prediction, target, delta=1.0, reduction='sum')
            count += len(target)
    finally:
        del stream
    return (total / count).item() if count else None

print(f'{TOTAL_POSITIONS:,} sampled positions in {INTERVALS} intervals; '
      f'validate and save every {POSITIONS_PER_INTERVAL:,} positions')


In [ ]:
# Fresh weights. Two GPUs, AdamW, Huber loss, gradient clipping and mixed precision.
from dataclasses import asdict
from time import perf_counter

config = RayGNNConfig(float32_reductions=True)
model = RayGNN(config).to(DEVICE)
network = torch.nn.DataParallel(model, device_ids=list(range(GPU_COUNT)))
print('Training with', GPU_COUNT, 'GPUs; global batch size', BATCH_SIZE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULE_INTERVALS)
scaler = torch.amp.GradScaler('cuda')
train_stream = make_stream(TRAIN_BINPACK, cyclic=True)
best = float('inf')
training_start = perf_counter()

try:
    for interval in range(INTERVALS):
        network.train()
        total, count = torch.zeros((), device=DEVICE), 0
        data_seconds = 0.0
        interval_start = perf_counter()
        for step in range(STEPS_PER_INTERVAL):
            data_start = perf_counter()
            sample = encode_scored_batch(*next(train_stream))
            data_seconds += perf_counter() - data_start
            if sample is None:
                continue
            batch, target = sample
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                prediction = network(batch.piece, batch.side_to_move, batch.castling, batch.en_passant)
                loss = F.huber_loss(prediction, target, delta=1.0)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += loss.detach() * len(target)
            count += len(target)
        for gpu_id in range(GPU_COUNT):
            torch.cuda.synchronize(gpu_id)
        train_seconds = perf_counter() - interval_start
        scheduler.step()
        train_loss = (total / count).item()
        val_loss = validation_huber(network)
        positions_seen = (interval + 1) * POSITIONS_PER_INTERVAL
        elapsed_hours = (perf_counter() - training_start) / 3600
        remaining_hours = elapsed_hours * (TOTAL_POSITIONS / positions_seen - 1)
        peak_gpu_gib = [round(torch.cuda.max_memory_allocated(i) / 2**30, 2)
                        for i in range(GPU_COUNT)]
        print(f'{positions_seen:,}/{TOTAL_POSITIONS:,} positions: train Huber={train_loss:.4f}' +
              (f', validation Huber={val_loss:.4f}' if val_loss is not None else '') +
              f', data/transfer={data_seconds / train_seconds:.0%} of train time' +
              f', peak GPU GiB={peak_gpu_gib}' +
              f', elapsed={elapsed_hours:.1f}h, estimated remaining={remaining_hours:.1f}h')
        checkpoint = {
            'model': model.state_dict(), 'config': asdict(config),
            'positions_seen': positions_seen, 'screening_interval': interval + 1,
            'target': 'White-positive pawn units = binpack score * side_to_move / 208',
            'input_schema': 'piece[64], side_to_move=+1/-1, castling=WK/WQ/BK/BQ, en_passant=0..64',
            'perspective': 'White-positive', 'train_binpack': str(TRAIN_BINPACK),
            'validation_huber': val_loss,
        }
        torch.save(checkpoint, RUN_DIR / 'last.pt')
        selection_loss = val_loss if val_loss is not None else train_loss
        if selection_loss < best:
            best = selection_loss
            torch.save(checkpoint, RUN_DIR / 'best.pt')
finally:
    del train_stream


In [ ]:
# Confirm that the saved model can be loaded by the engine adapter.
evaluator = RayGNNEvaluator.from_checkpoint(RUN_DIR / 'best.pt', DEVICE)
print('Starting-position White-positive centipawns:', evaluator.evaluate_cp([chess.Board()]).item())
for path in sorted(RUN_DIR.glob('*.pt')):
    print(path, f'{path.stat().st_size / 2**20:.1f} MiB')
